# NN_Melkior_Feature_Analysis
Trains 5 identical-architecture NNs on different feature subsets and compares ROC/AUC, AMS significance, and score distributions.

| Model | Dropped features |
|-------|------------------|
| 0 — Control | none |
| 1 — High-bias | PRI_had_pt, PRI_jet_subleading_pt, DER_mass_vis, DER_pt_ratio_lep_had |
| 2 — Phi (low-correlation) | PRI_lep_phi, PRI_had_phi, PRI_jet_subleading_phi, PRI_met_phi |
| 3 — Group 3 | PRI_jet_all_pt, DER_pt_h, DER_deltar_had_lep, DER_sum_pt |
| 4 — Group 4 | PRI_lep_eta, PRI_jet_all_pt, DER_deltar_had_lep, DER_pt_ratio_lep_had |

In [ ]:
!pip install HiggsML

In [ ]:
import os
import joblib
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
tf.config.run_functions_eagerly(False)

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.metrics import AUC
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler


class NeuralNetwork:

    def __init__(self, n_features=None):
        self.model  = None
        self.scaler = StandardScaler()

        self._predictions  = None
        self._test_labels  = None
        self._test_weights = None

        if n_features is not None:
            self._initialize_model(n_features)

    def _initialize_model(self, n_features):
        """Initialize the model architecture."""
        self.model = Sequential([
            Dense(256, input_dim=n_features, activation="relu"),
            BatchNormalization(), Dropout(0.3),

            Dense(256, activation="relu"),
            BatchNormalization(), Dropout(0.3),

            Dense(128, activation="relu"),
            BatchNormalization(), Dropout(0.3),

            Dense(128, activation="relu"),
            BatchNormalization(), Dropout(0.3),

            Dense(64, activation="relu"),
            BatchNormalization(), Dropout(0.3),

            Dense(1, activation="sigmoid"),
        ])

        self.model.compile(
            optimizer=Adam(learning_rate=1e-3),
            loss="binary_crossentropy",
            # AUC metric here is unweighted — same scale as val_auc.
            # Training loss is still computed with sample_weight (balanced),
            # so the optimiser still sees the rebalanced signal.
            metrics=[AUC(name="auc")],
        )

    def fit(self, train_data, y_train, weights_train=None):
        """Train the model."""
        if self.model is None:
            raise ValueError(
                "Model is not initialized. Ensure `_initialize_model` is called or load a saved model."
            )

        X_train = self.scaler.fit_transform(train_data)

        callbacks = [
            EarlyStopping(
                monitor="val_auc", mode="max",
                patience=10, restore_best_weights=True, verbose=1,
            ),
            ReduceLROnPlateau(
                monitor="val_auc", mode="max",
                factor=0.5, patience=5, min_lr=1e-6, verbose=1,
            ),
        ]

        self.history = self.model.fit(
            X_train, y_train,
            sample_weight=weights_train,
            epochs=100,
            batch_size=512,
            validation_split=0.2,
            callbacks=callbacks,
            verbose=2,
        )
        # Recompute unweighted train AUC after training so learning curve
        # shows train and val on the same unweighted scale.
        from sklearn.metrics import roc_auc_score as _ras
        train_preds = self.model.predict(X_train, verbose=0).ravel()
        self.auc_train_unweighted = float(_ras(y_train, train_preds))

    def predict(self, test_data, labels=None, weights=None):
        self._predictions = self.model.predict(
            self.scaler.transform(test_data), verbose=0
        ).ravel()

        if labels  is not None: self._test_labels  = np.asarray(labels)
        if weights is not None: self._test_weights = np.asarray(weights)

        return self._predictions

    def significance(self, test_labels=None, test_weights=None, total_events=None):
        if test_labels  is not None: self._test_labels  = np.asarray(test_labels)
        if test_weights is not None: self._test_weights = np.asarray(test_weights)

        if self._predictions is None:
            raise ValueError("No predictions found. Call predict() first.")
        if self._test_labels is None:
            raise ValueError(
                "True labels for test data are not available. Please provide them when calling predict()."
            )

        # Official AMS formula from the HiggsML starting kit:
        #   AMS(s, b) = sqrt(2 * ((s + b + bReg) * log(1 + s/(b+bReg)) - s))
        # bReg = 10 is a regularisation term that stabilises the formula at low b,
        # replacing the ad-hoc b>=1 guard we had before.
        # wFactor rescales weights from the test subset back to the full dataset scale,
        # exactly as the original starting kit does:
        #   wFactor = numPoints / numPointsValidation
        B_REG = 10.0

        def __ams(s, b):
            s = np.asarray(s, float)
            b = np.asarray(b, float)
            val = np.sqrt(2 * ((s + b + B_REG) * np.log(1 + s / (b + B_REG)) - s))
            val = np.where(s < 0, np.nan, val)
            if np.isscalar(s):
                return float(val)
            return val

        def __significance_vscore(y_true, y_score, sample_weight, w_factor):
            sample_weight = np.asarray(sample_weight)
            bins = np.linspace(0, 1.0, 101)
            s_hist, _ = np.histogram(
                y_score[y_true == 1], bins=bins, weights=sample_weight[y_true == 1]
            )
            b_hist, _ = np.histogram(
                y_score[y_true == 0], bins=bins, weights=sample_weight[y_true == 0]
            )
            s_cumul = np.cumsum(s_hist[::-1])[::-1] * w_factor
            b_cumul = np.cumsum(b_hist[::-1])[::-1] * w_factor
            return __ams(s_cumul, b_cumul)

        n_test = len(self._test_labels)
        w_factor = (total_events / n_test) if total_events is not None else 1.0

        ams_curve = __significance_vscore(
            y_true=self._test_labels,
            y_score=self._predictions,
            sample_weight=self._test_weights,
            w_factor=w_factor,
        )

        plt.plot(np.linspace(0, 1.0, 100), ams_curve, label="AMS Significance")
        plt.xlabel("Score threshold")
        plt.ylabel("AMS")
        return float(np.nanmax(ams_curve))

    def plot_learning_curves(self, weighted_test_auc=None, unweighted_test_auc=None):
        if not hasattr(self, "history"):
            raise ValueError("Model must be trained before plotting learning curves.")
        fig, ax1 = plt.subplots(figsize=(10, 5))
        ax2 = ax1.twinx()

        # val_auc from Keras is unweighted (Keras ignores sample_weight for val metrics).
        # history["auc"] is weighted (balanced weights used during training).
        # To make train vs val comparable we show the unweighted train AUC as a dot
        # at the final epoch, alongside the val_auc curve.
        l2, = ax2.plot(self.history.history["loss"],    color="tab:orange", label="Loss (train)")
        lines = [l2]
        if "val_auc" in self.history.history:
            l3, = ax1.plot(self.history.history["val_auc"], color="tab:blue", linestyle="--",
                           label="AUC (internal val, unweighted)")
            lines.append(l3)
        if "val_loss" in self.history.history:
            l4, = ax2.plot(self.history.history["val_loss"], color="tab:orange", linestyle="--",
                           label="Loss (internal val)")
            lines.append(l4)
        if hasattr(self, "auc_train_unweighted"):
            l1 = ax1.axhline(self.auc_train_unweighted, color="tab:blue", linestyle="-", linewidth=1.2,
                             label=f"AUC (train, unweighted) = {self.auc_train_unweighted:.4f}")
            lines.append(l1)
        if unweighted_test_auc is not None:
            l5 = ax1.axhline(unweighted_test_auc, color="tab:green", linestyle=":", linewidth=1.5,
                             label=f"AUC (test, unweighted) = {unweighted_test_auc:.4f}")
            lines.append(l5)
        if weighted_test_auc is not None:
            l6 = ax1.axhline(weighted_test_auc, color="tab:red", linestyle=":", linewidth=1.5,
                             label=f"AUC (test, weighted) = {weighted_test_auc:.4f}")
            lines.append(l6)

        ax1.set_xlabel("Epochs")
        ax1.set_ylabel("AUC",  color="tab:blue")
        ax2.set_ylabel("Loss", color="tab:orange")
        ax1.tick_params(axis="y", labelcolor="tab:blue")
        ax2.tick_params(axis="y", labelcolor="tab:orange")
        ax1.legend(lines, [l.get_label() for l in lines])
        ax1.grid(True)
        plt.title("Learning Curves — all AUC values unweighted for comparability")
        plt.tight_layout()
        plt.show()

    def plot_score_distribution(self, X_test, y_test):
        y_pred = self.predict(X_test)

        signal_scores = y_pred[y_test == 1]
        bkg_scores    = y_pred[y_test == 0]

        plt.figure(figsize=(8, 6))
        plt.hist(signal_scores, bins=50, alpha=0.5, label='Signal',     color='blue', density=True)
        plt.hist(bkg_scores,    bins=50, alpha=0.5, label='Background', color='red',  density=True)
        plt.title('Score Distribution (Signal vs Background)')
        plt.xlabel('Prediction Score')
        plt.ylabel('Density')
        plt.legend()
        plt.grid(True)
        plt.show()

    def save_model(self, path):
        """Save the trained model and scaler to the specified path."""
        os.makedirs(path, exist_ok=True)
        self.model.save(os.path.join(path, "model.keras"))
        joblib.dump(self.scaler, os.path.join(path, "scaler.pkl"))
        print(f"Model saved to {path}")

    def load_model(self, path):
        """Load the trained model and scaler from the specified path."""
        self.model  = load_model(os.path.join(path, "model.keras"))
        self.scaler = joblib.load(os.path.join(path, "scaler.pkl"))
        print(f"Model loaded from {path}")

### Load data

In [ ]:
import pandas as pd
from HiggsML.datasets import download_dataset

data = download_dataset("blackSwan_data")

# --- Train set (random sampling, intentionally non-deterministic) ---
data.load_train_set()
df_train = data.get_train_set()

ALL_FEATURES = [c for c in df_train.columns
                if c not in ("labels", "weights", "detailed_labels")]

y_train       = df_train["labels"].values
weights_train = df_train["weights"].values

# --- Deterministic test set ---
data.load_test_set()
test_set = data.get_test_set()

signal_df = test_set["htautau"].copy(); signal_df["labels"] = 1
bkg_frames = []
for proc in ("ztautau", "diboson", "ttbar"):
    df = test_set[proc].copy(); df["labels"] = 0
    bkg_frames.append(df)
df_test = pd.concat([signal_df] + bkg_frames, ignore_index=True)

y_test       = df_test["labels"].values
weights_test = df_test["weights"].values

# --- Rescale weights to match known physical LHC event yields ---
# From https://zenodo.org/records/15131565:
#   Signal: 1015 expected LHC events
#   Background: 1,002,395 + 3,783 + 44,192 = 1,050,370 expected LHC events
SUM_W_SIG = 1_015.0
SUM_W_BKG = 1_050_370.0

def rescale_weights(weights, labels, sum_sig, sum_bkg):
    w = weights.copy().astype(float)
    sig_mask = labels == 1
    bkg_mask = labels == 0
    if w[sig_mask].sum() > 0:
        w[sig_mask] *= sum_sig / w[sig_mask].sum()
    if w[bkg_mask].sum() > 0:
        w[bkg_mask] *= sum_bkg / w[bkg_mask].sum()
    return w

weights_train = rescale_weights(weights_train, y_train, SUM_W_SIG, SUM_W_BKG)
weights_test  = rescale_weights(weights_test,  y_test,  SUM_W_SIG, SUM_W_BKG)

print(f"Test sum W signal: {weights_test[y_test==1].sum():.1f}  (expected ~1015)")
print(f"Test sum W bkg:    {weights_test[y_test==0].sum():.1f}  (expected ~1050370)")

# --- Rebalance training weights ---
w_sig = weights_train[y_train == 1].sum()
w_bkg = weights_train[y_train == 0].sum()
weights_train_balanced = weights_train.copy()
weights_train_balanced[y_train == 1] *= w_bkg / w_sig
print(f"Weight scale applied to signal: {w_bkg / w_sig:.1f}x")
print(f"Train: {len(y_train)} events | Test: {len(y_test)} events | All features: {len(ALL_FEATURES)}")


### Define feature subsets

In [ ]:
# Each entry: (label, list of features to DROP)
FEATURE_GROUPS = [
    ("0 — Control (all features)",      []),
    ("1 — Drop high-bias",               ["PRI_had_pt", "PRI_jet_subleading_pt", "DER_mass_vis", "DER_pt_ratio_lep_had"]),
    ("2 — Drop phi (low-correlation)",   ["PRI_lep_phi", "PRI_had_phi", "PRI_jet_subleading_phi", "PRI_met_phi"]),
    ("3 — Drop group 3",                 ["PRI_jet_all_pt", "DER_pt_h", "DER_deltar_had_lep", "DER_sum_pt"]),
    ("4 — Drop group 4",                 ["PRI_lep_eta", "PRI_jet_all_pt", "DER_deltar_had_lep", "DER_pt_ratio_lep_had"]),
]

for label, dropped in FEATURE_GROUPS:
    kept = [f for f in ALL_FEATURES if f not in dropped]
    print(f"{label}: {len(kept)} features ({len(dropped)} dropped)")

### Train all 5 models

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

results = []  # list of dicts, one per model

for label, dropped in FEATURE_GROUPS:
    print(f"\n{'='*60}")
    print(f"Training: {label}")
    kept_cols = [f for f in ALL_FEATURES if f not in dropped]

    X_tr = df_train[kept_cols].values
    X_te = df_test[kept_cols].values

    nn = NeuralNetwork(n_features=len(kept_cols))
    nn.fit(X_tr, y_train, weights_train=weights_train_balanced)

    preds = nn.predict(X_te, labels=y_test, weights=weights_test)

    auc_w  = roc_auc_score(y_test, preds, sample_weight=weights_test)
    auc_u  = roc_auc_score(y_test, preds)
    ams    = nn.significance()
    plt.close()  # suppress inline plot from significance()

    fpr_w, tpr_w, _ = roc_curve(y_test, preds, sample_weight=weights_test)
    fpr_u, tpr_u, _ = roc_curve(y_test, preds)

    results.append({
        "label":    label,
        "dropped":  dropped,
        "n_feat":   len(kept_cols),
        "nn":       nn,
        "preds":    preds,
        "auc_w":    auc_w,
        "auc_u":    auc_u,
        "ams":      ams,
        "fpr_w":    fpr_w, "tpr_w": tpr_w,
        "fpr_u":    fpr_u, "tpr_u": tpr_u,
    })
    print(f"  AUC (weighted): {auc_w:.4f} | AUC (unweighted): {auc_u:.4f} | Max AMS: {ams:.4f}")

### Summary table

In [ ]:
print(f"{'Model':<35} {'Features':>8} {'AUC (w)':>10} {'AUC (u)':>10} {'Max AMS':>10}")
print("-" * 75)
for r in results:
    marker = " ← best" if r["ams"] == max(x["ams"] for x in results) else ""
    print(f"{r['label']:<35} {r['n_feat']:>8} {r['auc_w']:>10.4f} {r['auc_u']:>10.4f} {r['ams']:>10.4f}{marker}")

### ROC curves (weighted and unweighted)

In [ ]:
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
for r, c in zip(results, colors):
    ax1.plot(r["fpr_w"], r["tpr_w"], color=c, label=f"{r['label']} ({r['auc_w']:.4f})")
    ax2.plot(r["fpr_u"], r["tpr_u"], color=c, label=f"{r['label']} ({r['auc_u']:.4f})")
for ax, title in [(ax1, "ROC — Weighted AUC"), (ax2, "ROC — Unweighted AUC")]:
    ax.plot([0,1],[0,1],"k--",linewidth=0.8)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right", fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### AMS significance curves

In [ ]:
B_REG = 10.0
thresholds = np.linspace(0, 1, 100)
n_test = len(y_test)
w_factor = TOTAL_EVENTS / n_test

plt.figure(figsize=(10, 6))
for r, c in zip(results, colors):
    preds = r["preds"]
    bins = np.linspace(0, 1, 101)
    s_hist, _ = np.histogram(preds[y_test==1], bins=bins, weights=weights_test[y_test==1])
    b_hist, _ = np.histogram(preds[y_test==0], bins=bins, weights=weights_test[y_test==0])
    s_c = np.cumsum(s_hist[::-1])[::-1] * w_factor
    b_c = np.cumsum(b_hist[::-1])[::-1] * w_factor
    ams = np.where(s_c >= 0,
                   np.sqrt(2*((s_c+b_c+B_REG)*np.log(1+s_c/(b_c+B_REG))-s_c)),
                   np.nan)
    plt.plot(thresholds, ams, color=c, label=f"{r['label']} (max={r['ams']:.4f})")
plt.xlabel("Score threshold")
plt.ylabel("AMS")
plt.title("AMS significance vs. threshold — all feature subsets")
plt.legend(fontsize=8)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### Score distributions (Signal vs Background)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, (r, c) in enumerate(zip(results, colors)):
    ax = axes[i]
    ax.hist(r["preds"][y_test==1], bins=50, alpha=0.5, color="blue",  density=True, label="Signal")
    ax.hist(r["preds"][y_test==0], bins=50, alpha=0.5, color="red",   density=True, label="Background")
    ax.set_title(r["label"], fontsize=9)
    ax.set_xlabel("Score")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.4)
axes[-1].set_visible(False)  # 5 models, 6 subplots
plt.suptitle("Score distributions — Signal vs Background", fontsize=12)
plt.tight_layout()
plt.show()

### AUC and AMS comparison (bar charts)

In [ ]:
labels_short = [f"M{i}" for i in range(len(results))]
auc_w_vals = [r["auc_w"] for r in results]
auc_u_vals = [r["auc_u"] for r in results]
ams_vals   = [r["ams"]   for r in results]
x = np.arange(len(results))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# AUC
w_bars = ax1.bar(x - 0.2, auc_w_vals, 0.35, label="Weighted AUC",   color="tab:blue",   alpha=0.8)
u_bars = ax1.bar(x + 0.2, auc_u_vals, 0.35, label="Unweighted AUC", color="tab:orange", alpha=0.8)
ax1.set_xticks(x); ax1.set_xticklabels(labels_short)
ax1.set_ylabel("AUC")
mean_auc = np.mean(auc_w_vals)
margin = max(np.std(auc_w_vals) * 4, 0.005)
ax1.set_ylim(max(0, mean_auc - margin), min(1, mean_auc + margin))
ax1.set_title("AUC per feature subset")
ax1.legend()
ax1.grid(axis="y", linestyle="--", alpha=0.5)
for bar, val in zip(w_bars, auc_w_vals):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f"{val:.4f}", ha="center", va="bottom", fontsize=7)
for bar, val in zip(u_bars, auc_u_vals):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f"{val:.4f}", ha="center", va="bottom", fontsize=7)

# AMS
ams_bars = ax2.bar(x, ams_vals, color=colors, alpha=0.8)
ax2.set_xticks(x); ax2.set_xticklabels(labels_short)
ax2.set_ylabel("Max AMS")
mean_ams = np.mean(ams_vals)
margin_ams = max(np.std(ams_vals) * 4, 0.05)
ax2.set_ylim(max(0, mean_ams - margin_ams), mean_ams + margin_ams)
ax2.set_title("Max AMS per feature subset")
ax2.grid(axis="y", linestyle="--", alpha=0.5)
for bar, val in zip(ams_bars, ams_vals):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f"{val:.4f}", ha="center", va="bottom", fontsize=8)

# Legend for AMS bars
for bar, r in zip(ams_bars, results):
    bar.set_label(r["label"])
ax2.legend(fontsize=7, loc="lower right")

plt.tight_layout()
plt.show()